---

<div align="center">
  <img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/python/python-original.svg" width="80"/>
</div>

<h1 align="center">Aquisição: reviews da Amazon</h1>

<h3 align="center">Ângelo Rosa</h3>

<div align="center">
  <img src="https://img.shields.io/badge/Python-3776AB?style=for-the-badge&logo=python&logoColor=white"/>
  <img src="https://img.shields.io/badge/Jupyter-F37626?style=for-the-badge&logo=jupyter&logoColor=white"/>
</div>

---

## 1. Contexto e objetivo

Baixar os reviews da categoria `All_Beauty` do *Amazon Reviews 2023* e gravar em `data/reviews.db`,
o banco que todos os módulos seguintes consomem.

O recorte é **uma categoria, um ano inteiro, sem amostragem**: todos os reviews de 2021. Isso tem
três consequências boas:

- **Não há desenho amostral para justificar.** O conjunto é o ano inteiro daquela categoria, então
  qualquer estatística calculada aqui descreve a população, não uma amostra dela.
- **A leitura temporal fica completa.** Os 12 meses estão inteiros, e não por aproximação.
- **A classe neutra ganha volume.** Ela é a difícil do módulo 2, e passa de centenas para milhares
  de exemplos.

O preço é perder a comparação entre categorias. Fica valendo a comparação dentro da categoria: meses,
compras verificadas contra não verificadas, produtos diferentes.

Os únicos filtros são de qualidade: texto com pelo menos 20 caracteres e sem duplicatas de
(usuário, texto), que são copy-paste e spam.

In [1]:
import os
import sqlite3

import pandas as pd
from huggingface_hub import hf_hub_download as hf
from tqdm.auto import tqdm

from config import (ANO_ALVO, CATEGORIA, DB_PATH, META_COLS, MIN_CARACTERES,
                    REPO_ID, REVIEW_COLS)
from utils.sql import query_sql

c:\Users\PC\Documents\Projetos\amazon-reviews-chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Download dos arquivos brutos

Dois `.jsonl`, um de reviews e um de metadados dos produtos. O `hf_hub_download` guarda em
`data/hf_cache`, então rodar de novo não baixa outra vez.

In [2]:
def baixar(pasta: str, nome: str) -> str:
    return hf(repo_id=REPO_ID, repo_type="dataset",
              filename=f"raw/{pasta}/{nome}.jsonl", cache_dir="data/hf_cache")


arquivo_reviews = baixar("review_categories", CATEGORIA)
arquivo_meta = baixar("meta_categories", f"meta_{CATEGORIA}")
print("arquivos prontos")

c:\Users\PC\Documents\Projetos\amazon-reviews-chatbot\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PC\Documents\Projetos\amazon-reviews-chatbot\data\hf_cache. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\PC\Documents\Projetos\amazon-reviews-chatbot\.venv\Lib\site-packages\huggingfac

arquivos prontos


## 3. Reviews do ano

O arquivo tem todos os reviews da categoria desde os anos 2000, então ele é lido em pedaços e o
filtro de ano é aplicado logo em seguida.

In [3]:
def jsonl_to_frame(src: str, cols: list[str]) -> pd.DataFrame:
    """Lê um .jsonl em chunks e devolve um DataFrame só com as colunas pedidas."""
    with pd.read_json(src, lines=True, chunksize=100_000) as reader:
        pedacos = [chunk[cols] for chunk in tqdm(reader, desc=os.path.basename(src))]
    return pd.concat(pedacos, ignore_index=True)


reviews = jsonl_to_frame(arquivo_reviews, REVIEW_COLS)
print(f"reviews na categoria: {len(reviews):,}")

# o parent_asin agrupa as variacoes do mesmo produto e e o que casa com os metadados
reviews = reviews.rename(columns={"parent_asin": "product_id", "timestamp": "review_date"})
reviews["review_date"] = pd.to_datetime(reviews["review_date"], unit="ms")

reviews = reviews[(reviews["review_date"].dt.year == ANO_ALVO)
                  & (reviews["text"].str.len() >= MIN_CARACTERES)]
reviews = reviews.drop_duplicates(subset=["user_id", "text"])

reviews["review_date"] = reviews["review_date"].dt.strftime("%Y-%m-%d %H:%M:%S")
reviews["verified_purchase"] = reviews["verified_purchase"].astype(int)
reviews["helpful_vote"] = reviews["helpful_vote"].fillna(0).astype(int)

reviews = reviews.sort_values("review_date").reset_index(drop=True)
reviews.insert(0, "review_id", range(1, len(reviews) + 1))
print(f"reviews em {ANO_ALVO} após filtros: {len(reviews):,}")
reviews.head(3)

All_Beauty.jsonl: 8it [00:07,  1.10it/s]


reviews na categoria: 701,528
reviews em 2021 após filtros: 111,851


,review_id,rating,title,text,asin,product_id,user_id,review_date,helpful_vote,verified_purchase
0,1,5,exquisite,Beautiful brush collection.,B0895NCNF8,B0895NCNF8,AGFO56M4RXP2ANMHWINQ5BDHTVLA,2021-01-01 00:02:49,0,1
1,2,5,Easy for beginners and update: bought 2 more,I'm very new at magnetic eyelashes and had bou...,B086616RTT,B086616RTT,AGR4NL7ZS5WOPJNQGVTYAWWIA36A,2021-01-01 00:04:29,0,1
2,3,4,I like eye patches,They are so relaxing and you can leave them on...,B08KWN77LW,B08KWN77LW,AG73BVBKUOH22USSFJA5ZWL7AKXA,2021-01-01 00:12:16,0,0


## 4. Metadados dos produtos

Uma linha por `product_id`, que é o que dá título, preço e loja aos reviews na hora do `JOIN`.

In [4]:
products = jsonl_to_frame(arquivo_meta, META_COLS)
products = (products.rename(columns={"parent_asin": "product_id"})
            .assign(price=lambda d: pd.to_numeric(d["price"], errors="coerce"))
            .drop_duplicates(subset="product_id"))
print(f"produtos no metadado: {len(products):,}")
products.head(3)

meta_All_Beauty.jsonl: 0it [00:00, ?it/s]

meta_All_Beauty.jsonl: 2it [00:03,  1.84s/it]

produtos no metadado: 112,590


,product_id,title,main_category,average_rating,rating_number,price,store
0,B01CUPMQZE,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",All Beauty,4.8,10,NaN,Howard Products
1,B076WQZGPM,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,All Beauty,4.5,3,NaN,Yes To
2,B000B658RI,Eye Patch Black Adult with Tie Band (6 Per Pack),All Beauty,4.4,26,NaN,Levine Health Products


## 5. Banco SQLite

O `DROP TABLE IF EXISTS` deixa a célula reexecutável: sem ele, rodar o notebook uma segunda vez
falha porque as tabelas já existem.

In [5]:
conn = sqlite3.connect(DB_PATH)
conn.executescript("""
DROP TABLE IF EXISTS reviews;
DROP TABLE IF EXISTS products;

CREATE TABLE products (
    product_id     TEXT PRIMARY KEY,
    title          TEXT,
    main_category  TEXT,
    average_rating REAL,
    rating_number  INTEGER,
    price          REAL,
    store          TEXT
);

CREATE TABLE reviews (
    review_id         INTEGER PRIMARY KEY,
    product_id        TEXT NOT NULL REFERENCES products(product_id),
    user_id           TEXT NOT NULL,
    asin              TEXT,
    rating            REAL NOT NULL,
    title             TEXT,
    text              TEXT,
    review_date       TEXT NOT NULL,
    helpful_vote      INTEGER,
    verified_purchase INTEGER
);

CREATE INDEX idx_reviews_product ON reviews(product_id);
CREATE INDEX idx_reviews_date    ON reviews(review_date);
""")

products.to_sql("products", conn, if_exists="append", index=False)
reviews.to_sql("reviews", conn, if_exists="append", index=False)
conn.commit()
print("banco gravado")

banco gravado


## 6. Conferência

O resumo do ano e a cobertura mês a mês, que é o que o recorte de um ano inteiro garante.

In [6]:
resumo = query_sql(conn, """
SELECT COUNT(*)                                 AS n_reviews,
       ROUND(AVG(rating), 2)                    AS nota_media,
       ROUND(100.0 * AVG(verified_purchase), 1) AS pct_verificadas,
       MIN(DATE(review_date))                   AS primeira,
       MAX(DATE(review_date))                   AS ultima
FROM reviews
""")
print(resumo.to_string(index=False))

por_mes = query_sql(conn, """
SELECT strftime('%m', review_date) AS mes, COUNT(*) AS n, ROUND(AVG(rating), 2) AS nota
FROM reviews GROUP BY mes
""")
conn.close()
por_mes

 n_reviews  nota_media  pct_verificadas   primeira     ultima
    111851        3.73             91.2 2021-01-01 2021-12-31


,mes,n,nota
0,01,11768,3.91
1,02,10554,3.80
2,03,12452,3.81
3,04,11373,3.79
4,05,10124,3.75
5,06,10093,3.70
6,07,9818,3.71
7,08,8617,3.66
8,09,7543,3.61
9,10,7274,3.58
